In [54]:
import polars as pl
import re

In [4]:
DP_CFG = pl.read_excel("gsat7rTcPlugin/data.xlsx",sheet_name="CFG")
DP_RT_ADDRESS = pl.read_excel("gsat7rTcPlugin/data.xlsx",sheet_name="RT_ADDRESS")
NCO1_FREQ = pl.read_excel("gsat7rTcPlugin/data.xlsx",sheet_name="CH_NCO1")


In [5]:
NCO1_FREQ

CHANNELIZER_IP_FREQUENCIES,FREQUENCY,CODE
str,f64,str
"""f1""",292.5,"""001"""
"""f2""",297.5,"""010"""
"""f3""",302.5,"""011"""
"""f4""",307.5,"""100"""
"""f5""",312.5,"""101"""
"""f6""",317.5,"""110"""


In [6]:
NCO1 = DP_CFG.filter(pl.col("CONFIG") == 1).select(pl.col(["SNO","CONFIG","CHANNELIZER_IP_PORTS","CHANNELIZER_IP_FREQUENCIES"]))

In [7]:
NCO1 = NCO1.join(NCO1_FREQ,on="CHANNELIZER_IP_FREQUENCIES")
NCO1

SNO,CONFIG,CHANNELIZER_IP_PORTS,CHANNELIZER_IP_FREQUENCIES,FREQUENCY,CODE
i64,i64,str,str,f64,str
1,1,"""m1_1""","""f1""",292.5,"""001"""
2,1,"""m1_2""","""f2""",297.5,"""010"""
3,1,"""m2_1""","""f3""",302.5,"""011"""
4,1,"""m2_2""","""f4""",307.5,"""100"""
5,1,"""r1_1""","""f1""",292.5,"""001"""
6,1,"""r1_2""","""f2""",297.5,"""010"""
7,1,"""r2_1""","""f3""",302.5,"""011"""
8,1,"""r2_2""","""f4""",307.5,"""100"""


In [10]:
def generate_ch_nco1_cmd():
    address = 5
    rt_address = DP_RT_ADDRESS.filter(pl.col("SUB_ADDRESS") == address)["ADDRESS"][0].zfill(4)
    s = "".join(NCO1["CODE"].to_list())
    print(s)
    data_bytes = [ hex(int(s[i:i+4], 2))[2:].zfill(2) for i in range(0, len(s), 4)]
    cmd = ['sendtcp', '1553trantmtcbus',rt_address]
    cmd.extend(data_bytes)
    return cmd
generate_ch_nco1_cmd()

001010011100001010011100


['sendtcp', '1553trantmtcbus', '0065', '02', '09', '0c', '02', '09', '0c']

In [48]:
data_bytes.insert(0,)


In [49]:

data_bytes

[['sendtcp', '1553trantmtcbus'], '0065', '02', '09', '0c', '02', '09', '0c']

In [45]:
data_bytes

['0065', '02', '09', '0c', '02', '09', '0c']

In [30]:
hex()

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1966302147.py, line 1)

In [ ]:
def generate_ch_ip_ports_command(cfg_number):
    

In [32]:
def make_ch_2nd_nco_word(channel: int, nco_value: int) -> str:
    """
    Generate a 16-bit binary string for the NCO word.
    channel: int (0-7), will be placed in bits 11-9
    nco_value: int (0-511), will be placed in bits 8-0
    Returns: string, e.g., '0001000100001011'
    """
    # Limit channel and nco_value to their ranges
    channel = channel & 0b111       # 3 bits
    nco_value = nco_value & 0x1FF   # 9 bits

    word = (channel << 9) | nco_value
    # Format as 16-bit binary string
    return f"{word:016b}"

# Example usage:
print(make_nco_word(channel=5, nco_value=123))  # e.g., '0001011111011'


0000101001111011


In [33]:
def generate_band_select_words(channelizer_freq_map):
    """
    channelizer_freq_map: dict with keys:
        m1_1, m1_2, m2_1, m2_2, r1_1, r1_2, r2_1, r2_2
        and values: frequency code as 'f1', 'f2', ..., or actual freq string

    Returns (band_select1_16bit, band_select2_16bit) as binary strings
    """
    FREQ_TO_CODE = {"292.5": 0b001,"297.5": 0b010,"302.5": 0b011,"307.5": 0b100, "312.5": 0b101, "317.5": 0b110}
    FREQ_CODE_MAP = {"f1": 0b001, "f2": 0b010,"f3": 0b011,"f4": 0b100,"f5": 0b101,"f6": 0b110}
    # Select your map depending on input:
    code_map = FREQ_CODE_MAP if list(channelizer_freq_map.values())[0].startswith('f') else FREQ_TO_CODE

    # BAND SELECT 1 bit assignment:
    # Bit  2-0: m1_1 (ADC-A, Ch1 Main)
    # Bit  5-3: m1_2 (ADC-B, Ch1 Main)
    # Bit  8-6: m2_1 (ADC-A, Ch2 Main)
    # Bit 11-9: m2_2 (ADC-B, Ch2 Main)
    # Bit 14-12: r1_1 (ADC-A, Ch3 Redundant-1)
    band_select1 = 0
    band_select1 |= (code_map[channelizer_freq_map["m1_1"]] & 0b111)         # bits 2-0
    band_select1 |= (code_map[channelizer_freq_map["m1_2"]] & 0b111) << 3    # bits 5-3
    band_select1 |= (code_map[channelizer_freq_map["m2_1"]] & 0b111) << 6    # bits 8-6
    band_select1 |= (code_map[channelizer_freq_map["m2_2"]] & 0b111) << 9    # bits 11-9
    band_select1 |= (code_map[channelizer_freq_map["r1_1"]] & 0b111) << 12   # bits 14-12

    # BAND SELECT 2 bit assignment:
    # Bit  2-0: r1_2 (ADC-B, Ch3 Redundant-1)
    # Bit  5-3: r2_1 (ADC-A, Ch4 Redundant-2)
    # Bit  8-6: r2_2 (ADC-B, Ch4 Redundant-2)
    band_select2 = 0
    band_select2 |= (code_map[channelizer_freq_map["r1_2"]] & 0b111)         # bits 2-0
    band_select2 |= (code_map[channelizer_freq_map["r2_1"]] & 0b111) << 3    # bits 5-3
    band_select2 |= (code_map[channelizer_freq_map["r2_2"]] & 0b111) << 6    # bits 8-6

    # Format as 16-bit binary string
    bs1_str = f"{band_select1:016b}"
    bs2_str = f"{band_select2:016b}"
    return bs1_str, bs2_str

# Example usage:
channelizer_map = dict(zip(
    NCO1["CHANNELIZER_IP_PORTS"].to_list(),
    NCO1["CHANNELIZER_IP_FREQUENCIES"].to_list()
))
print(generate_band_select_words(channelizer_map))

('0001100011010001', '0000000100011010')


In [30]:
s = f"{word:020b}"
s

'00000000101001111011'

In [50]:
def binary_to_hex_data_bytes(bin_str):
    # Ensure length is multiple of 4
    bin_str = bin_str.zfill((len(bin_str) + 3) // 4 * 4)
    # Convert to integer and format as uppercase hex
    s = f"{int(bin_str, 2):0{len(bin_str) // 4}X}"
    data_bytes = [s[i:i+2] for i in range(0, len(s), 2)]
    return data_bytes
# Example usage
bin_str = '0000101001111011'
hex_nibbles = binary_to_hex_data_bytes(bin_str)
hex_nibbles

['0A', '7B']

In [46]:
s="0A7B"
s[::2]

'07'

In [53]:
def comm_channel_code(ch_port: str) -> str:
    """
    ch_port: string like 'm1_1', 'm2_4', 'r1_8', etc.
    Returns a 5-bit binary code string, e.g. '00000' (for m1_1), '01101' (for m2_2)
    """
    channelizer_map = {
        "m1": 0b00,
        "m2": 0b01,
        "r1": 0b10,
        "r2": 0b11,
    }
    ch, port = ch_port.split("_")
    port_num = int(port)
    ch_bits = channelizer_map[ch]      # 2 bits
    port_bits = port_num - 1           # Port 1 = 0, Port 2 = 1, ... Port 8 = 7
    # Compose: [ch_bits (2)] [port_bits (3)]
    code = (ch_bits << 3) | port_bits
    return f"{code:016b}"

# Example usage:
for ex in ['m1_1', 'm2_2', 'r1_5', 'r2_8']:
    print(f"{ex}: {comm_channel_code(ex)}")

# Output:
# m1_1: 00000
# m2_2: 01001
# r1_5: 100100
# r2_8: 11111


m1_1: 0000000000000000
m2_2: 0000000000001001
r1_5: 0000000000010100
r2_8: 0000000000011111


In [55]:
inputs = ["B2_FL_S_M1_OP_P1", "B1_FL_S_M1_OP_P2", "B4_FL_S_M2_OP_P1", "B3_FL_S_M2_OP_P2"]
inputs.sort(key=lambda x: int(re.search(r'B(\d+)', x).group(1)))
inputs

['B1_FL_S_M1_OP_P2',
 'B2_FL_S_M1_OP_P1',
 'B3_FL_S_M2_OP_P2',
 'B4_FL_S_M2_OP_P1']

In [65]:

bits = ["1","0", "1", "1", "0", "0"]
current_bits = ["0"] * len(bits)
on_count = 0
for i, b in enumerate(bits):
    if b == "1":
        current_bits[i] = "1"
        on_count += 1
        if on_count > 3:
            break
        bits_str = "".join(current_bits[::-1])  # Reverse back to MSB->LSB for data word
        bits_str = bits_str.zfill(16)
        data_bytes = bits_str
        print(bits_str)
        



0000000000000001
0000000000000101
0000000000001101


In [79]:
import copy
bits = ["1", "0", "1", "1", "1", "0"] # Now bits[0] is LSB (FL_C_M), bits[5] is MSB (RL_C_R2)
current_bits = copy.deepcopy(bits)  # Start with all bits as they are
for i, b in enumerate(bits):
    if b == "1":
        if i == 0:
            continue
        current_bits[i] = "0"
        current_bits[0] = "1"
        bits_str = "".join(current_bits[::-1])  # Reverse back to MSB->LSB for data word
        bits_str = bits_str.zfill(16)
        print(bits_str)

0000000000011001
0000000000010001
0000000000000001
